# 07 — ZITboostGu (Gu 2024 논문충실) · 경량 Optuna + ζ profile likelihood · 후처리/iso

논문충실 **순수 ZIT**(`modules.zit_Gu.ZITboostGu`, bag 아님)를 die-level broadcast로 학습하고,
**경량 Optuna**로 HP를 1세트 찾은 뒤 **ζ는 Algorithm 2 profile likelihood**로 선택한다.
후처리(집계→zero_clip)·isotonic/tail 보정은 `06_bag_zit_eql_seed_sweep_final` 컨셉을 그대로 재사용.

## 핵심 설계
- 모델: `ZITboostGu` — π=cross_entropy, μ=tweedie, φ=gamma(EQL deviance, custom obj). die-level 학습(unit health를 4 die에 broadcast).
- die→unit 집계: **mean 계열**(plain ZIT이므로 sum 아님).
- **ζ를 먼저 결정** (논문 Algorithm 2): HPO 전에 anchor HP로 `ZETA_GRID`를 `score_loglik`(val) 최대로 골라 `ζ*` 확정. (ζ=바깥 파라미터)
- HPO: `objective = OOF unit RMSE`(mean 집계, **unit-grouped** fold). ζ는 위에서 정한 `ζ*`로 **고정**하고 외부 GBT HP만 흔든다.
- 최종: best (HP, ζ)로 **5-fold OOF** → 후처리(zero_clip) → isotonic/tail grid → val/test RMSE. (tau_pi π-게이트는 제거 — 논문 예측 (1-π)μ 유지)
- seed sweep 없음(1세트). **경향 확인용 one-shot**. 컴퓨팅 예산 = `HPO_DEADLINE`(timeout).


## 0. 환경 설정 / import

In [1]:
from pathlib import Path
import gc
import itertools
import json
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import runpy


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / 'setup.py').exists() and (cand / 'utils').exists():
            return cand
    raise RuntimeError('프로젝트 루트를 찾지 못함: setup.py + utils/ 기준')


ROOT = find_project_root()
runpy.run_path(str(ROOT / 'setup.py'))

from utils.config import (  # noqa: E402
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    OUTPUT_DIR,
    TARGET_COL,
    KEY_COL,
    DIE_KEY_COL,
    SEED as DEFAULT_SEED,
)
from utils.data import load_all, get_feat_cols, split_xs  # noqa: E402

PP_DIR = Path(CFG_PROJECT_ROOT) / '2_preprocessing'
if str(PP_DIR) not in sys.path:
    sys.path.insert(0, str(PP_DIR))
MOD_DIR = Path(CFG_PROJECT_ROOT) / '3_modeling'
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))

from meta_features import add_meta_features  # noqa: E402
from modules import preprocess, postprocess  # noqa: E402
from modules.zit_Gu import ZITboostGu  # noqa: E402
from sklearn.isotonic import IsotonicRegression  # noqa: E402
from sklearn.model_selection import KFold  # noqa: E402
from scipy.interpolate import PchipInterpolator  # noqa: E402
import optuna  # noqa: E402
optuna.logging.set_verbosity(optuna.logging.WARNING)

import logging  # noqa: E402
logging.getLogger('lightgbm').setLevel(logging.ERROR)

PROJECT_ROOT = Path(CFG_PROJECT_ROOT)
OUTPUT_DIR = Path(OUTPUT_DIR)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')


setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
OUTPUT_DIR   = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output


## 1. 실행 설정

In [2]:
# ===== 실행 설정 (경량 Optuna + ζ profile likelihood, 후처리/iso는 06 노트북 컨셉) =====
EXP_TAG = 'zit_gu_optuna'
RUN_TAG = datetime.now().strftime('run_%m%d_%H%M%S')
OUT_DIR = OUTPUT_DIR / '01_zit' / 'zit_gu' / EXP_TAG / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5          # 최종 OOF fold 수
HPO_N_FOLDS = 3      # HPO 동안은 3-fold (속도). OOF는 3-fold도 전 unit을 커버.
N_JOBS = -1
MODEL_NAME = 'zit_gu'

# --- HPO 예산: 데드라인 timeout. 컴퓨팅 자원이 내일 04:00까지이므로 HPO는 03:00에 끊고
#     남은 1시간을 ζ 그리드 + 최종 5-fold + 후처리에 쓴다. (now가 데드라인을 넘으면 최소 300초.)
HPO_DEADLINE = datetime(2026, 6, 9, 3, 0, 0)
HPO_MAX_TRIALS = 30         # 상한 (timeout이 먼저 걸리면 그 전에 종료)

# ζ profile likelihood (논문 Algorithm 2) 그리드. ζ를 HPO보다 '먼저' anchor HP로 결정한다.
#   하한을 1.05까지 내림 — 같은 데이터의 BagZIT study가 ζ≈1.05를 찾았기 때문(저분산/Poisson 근처).
ZETA_GRID = [1.4, 1.45, 1.5, 1.55, 1.6]

# 전처리: 06 노트북과 동일 파이프라인 (param1_005t17.json effective_pp_params 재사용)
PP_PARAMS = {
    'missing_threshold': 0.3,
    'corr_threshold': 0.9,
    'corr_keep_by': 'std',
    'add_indicator': True,
    'indicator_threshold': 0.05,
    'spatial_max_dist': 6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by': 'std',
}
CLIP_Y_EXTREME = True

# --- 후처리/iso 전역 (06 노트북 공통함수 셀이 그대로 참조). plain ZIT이라 die->unit은 mean 계열 ---
BASELINE_AGG = 'mean'
AGG_CANDIDATES = ('mean', 'median', 'trimmed_mean', 'Q75', 'max')
POSITION_METHOD = 'optuna'        # AGG_CANDIDATES에 'weighted' 없음 -> 실질 no-op
POSITION_OPTUNA_N_TRIALS = 30
ZERO_CLIP_RANGE = (0.0001, 0.003)
ZERO_CLIP_N = 30
ZERO_CLIP_LOG_SPACE = True
ISO_KINDS = ['step', 'pchip']
ISO_WEIGHTS = [0.25, 0.5, 0.75, 1.00, 1.25, 1.50]
TAIL_QS = [0.95, 0.975, 0.99]
TAIL_RESID_QS = [0.75, 0.90]
TAIL_GAINS = [0.0, 0.5, 1.0, 1.5, 2.5]
TAIL_POWERS = [1.0, 2.0]
IQR_TOP_KS = [0, 1, 2]
IQR_MARGIN = 1e-6

print(f'OUT_DIR          = {OUT_DIR}')
print(f'N_FOLDS={N_FOLDS}, HPO_N_FOLDS={HPO_N_FOLDS}')
print(f'HPO_DEADLINE     = {HPO_DEADLINE}  (now={datetime.now()})')
print(f'ZETA_GRID        = {ZETA_GRID}')


OUT_DIR          = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\zit_gu\zit_gu_optuna\run_0608_113938
N_FOLDS=5, HPO_N_FOLDS=3
HPO_DEADLINE     = 2026-06-09 03:00:00  (now=2026-06-08 11:39:38.905179)
ZETA_GRID        = [1.4, 1.45, 1.5, 1.55, 1.6]


## 2. 데이터 로드 + 전처리

In [3]:
# ===== 데이터 로드 + 전처리 + 메타피처 + die-level broadcast =====
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y 극단값 clipping (06 노트북과 동일 재현)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    yr = ys_input['train'][TARGET_COL]
    second_max = yr[yr < yr.max()].max()
    n_clip = int((yr >= yr.max()).sum())
    ys_input['train'][TARGET_COL] = yr.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] train max -> {second_max:.9f}, clipped={n_clip}')

# 전처리 (06 노트북과 동일 PP_PARAMS) + position/die_x/die_y 메타피처
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_PARAMS)
xs_train, xs_val, xs_test = pp['xs_train'], pp['xs_val'], pp['xs_test']
feat_cols_clean = pp['feat_cols']
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val = xs_val[feat_cols_clean].values.astype(np.float64)
X_test = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die = xs_val[KEY_COL].values
uid_test_die = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# die-level target: unit health를 그 unit의 4 die에 동일하게 broadcast (plain ZIT die-level 학습)
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

print(f'[data] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}')
print(f'[units] train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}, feat={len(feat_cols_clean)}')
print(f'[die target] zero_frac={(y_train_die == 0).mean():.3f}')


[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] train max -> 0.097417066, clipped=1
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 

## 3. 공통 함수 (후처리·isotonic/tail — 06 노트북 재사용)

In [4]:
# 공통 RMSE 계산. 모든 선택 기준은 validation RMSE가 낮은 쪽이다.
def rmse(pred, y):
    pred = np.asarray(pred, dtype=float)
    y = np.asarray(y, dtype=float)
    return float(np.sqrt(np.mean((pred - y) ** 2)))


def clip_nonneg(x):
    return np.clip(np.asarray(x, dtype=float), 0.0, None)


# ZIT의 structural-zero 확률 pi가 tau_pi보다 큰 die는 0으로 강제한다.
def apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def unit_rmse(unit_df, y_unit_s):
    p = unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values
    return rmse(p, y_unit_s.values)


def aligned_unit_pred(unit_df, y_unit_s):
    return unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values.astype(float)


def tune_unit_postprocess_train_val(
    xs_train,
    xs_val,
    xs_test,
    die_pred_train,
    die_pred_val,
    die_pred_test,
    y_train_unit_df,
    y_val_unit_df,
):
    """hp/002 seed sweep과 같은 train/validation 전용 축약판.

    차이: zero_clip 후보 array를 np.arange linear가 아니라 np.logspace로 만든다.
    하한 0.0001부터 상한 0.015까지 log 균등 30개. 작은 양수도 촘촘히 본다.
    """
    y_train_s = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]
    y_val_s = y_val_unit_df.set_index(KEY_COL)[TARGET_COL]
    decisions = {}
    val_history = []

    # 1단계: baseline 집계(mean)에서 출발. validation 개선 시에만 교체.
    train_unit = postprocess.aggregate(xs_train, die_pred_train, BASELINE_AGG)
    val_unit = postprocess.aggregate(xs_val, die_pred_val, BASELINE_AGG)
    test_unit = postprocess.aggregate(xs_test, die_pred_test, BASELINE_AGG)
    cur_val = unit_rmse(val_unit, y_val_s)
    val_history.append((f'baseline_{BASELINE_AGG}', cur_val))

    # 2단계: train OOF에서 best 집계 방식 후보, validation 개선 시에만 채택.
    agg_res = postprocess.find_best_aggregation(
        xs_train,
        die_pred_train,
        y_train_unit_df,
        methods=AGG_CANDIDATES,
        position_method=POSITION_METHOD,
        optuna_n_trials=POSITION_OPTUNA_N_TRIALS,
    )
    best_agg_cand = agg_res['best_method']
    pos_w_cand = agg_res['pos_weights']

    if best_agg_cand == BASELINE_AGG:
        best_agg = BASELINE_AGG
        pos_w = None
        decisions['aggregation'] = f'{BASELINE_AGG} train OOF best -> 유지'
    else:
        cand_train = postprocess.aggregate(xs_train, die_pred_train, best_agg_cand, pos_w_cand)
        cand_val = postprocess.aggregate(xs_val, die_pred_val, best_agg_cand, pos_w_cand)
        cand_test = postprocess.aggregate(xs_test, die_pred_test, best_agg_cand, pos_w_cand)
        cand_val_rmse = unit_rmse(cand_val, y_val_s)
        if cand_val_rmse < cur_val:
            train_unit, val_unit, test_unit = cand_train, cand_val, cand_test
            best_agg, pos_w = best_agg_cand, pos_w_cand
            decisions['aggregation'] = f'{best_agg_cand} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
            cur_val = cand_val_rmse
        else:
            best_agg, pos_w = BASELINE_AGG, None
            decisions['aggregation'] = f'{best_agg_cand} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append((f'after_agg({best_agg})', cur_val))

    # 3단계: zero_clip. hp/003 seed sweep의 핵심 변경 - log-spaced 후보.
    zc_arr = np.logspace(
        np.log10(ZERO_CLIP_RANGE[0]),
        np.log10(ZERO_CLIP_RANGE[1]),
        ZERO_CLIP_N,
    )
    zc_res = postprocess.find_best_zero_clip(train_unit, y_train_unit_df, zc_arr, log_space=ZERO_CLIP_LOG_SPACE)
    cand_zc = zc_res['best_threshold']
    cand_train = postprocess.apply_zero_clip(train_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val = postprocess.apply_zero_clip(val_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_test = postprocess.apply_zero_clip(test_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val_rmse = unit_rmse(cand_val, y_val_s)

    best_zc = None
    if cand_val_rmse < cur_val:
        train_unit, val_unit, test_unit = cand_train, cand_val, cand_test
        best_zc = cand_zc
        decisions['zero_clip'] = f'{cand_zc:.6f} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
        cur_val = cand_val_rmse
    else:
        decisions['zero_clip'] = f'{cand_zc:.6f} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append(('after_zero_clip', cur_val))

    train_rmse = unit_rmse(train_unit, y_train_s)
    return {
        'best_agg': best_agg,
        'pos_weights': pos_w,
        'best_zero_clip': best_zc,
        'zero_clip_log_space': ZERO_CLIP_LOG_SPACE,
        'zero_clip_arr': zc_arr,
        'position_method': POSITION_METHOD,
        'agg_rmses': agg_res['rmse_per_method'],
        'decisions': decisions,
        'val_rmse_history': val_history,
        'train_rmse': train_rmse,
        'val_rmse_final': cur_val,
        'final_train_unit': train_unit,
        'final_val_unit': val_unit,
        'final_test_unit': test_unit,
    }


def iqr_stats(pred, y_true=None):
    pred = np.asarray(pred, dtype=float)
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    mask = pred > upper
    out = {
        'q1': float(q1),
        'q3': float(q3),
        'iqr': float(iqr),
        'upper_fence': float(upper),
        'n_upper_outliers': int(mask.sum()),
        'max_pred': float(np.max(pred)),
    }
    if y_true is not None and mask.any():
        yy = np.asarray(y_true, dtype=float)[mask]
        out.update({
            'outlier_true_mean': float(np.mean(yy)),
            'outlier_true_max': float(np.max(yy)),
            'outlier_true_ge_q95': int((yy >= np.quantile(y_true, 0.95)).sum()),
        })
    else:
        out.update({'outlier_true_mean': np.nan, 'outlier_true_max': np.nan, 'outlier_true_ge_q95': 0})
    return out


def push_top_k_to_iqr(pred, score, top_k=0, margin=1e-6):
    """예측 rank만 사용하는 batch 변환. y_true는 절대 보지 않는다."""
    pred = np.asarray(pred, dtype=float).copy()
    if top_k <= 0:
        return pred
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    idx = np.argsort(np.asarray(score, dtype=float))[-int(top_k):]
    pred[idx] = np.maximum(pred[idx], upper + margin)
    return pred


def build_iso_pchip_transform(iso):
    """sklearn IsotonicRegression의 step function을 PCHIP monotonic cubic으로 smoothing한다.

    - knot 값(`X_thresholds_`, `y_thresholds_`)은 그대로 둔다. → 상단 끌어올림 폭은 step과 동일.
    - knot 사이만 PCHIP cubic 보간. → 평탄 plateau가 부드러운 곡선이 된다.
    - PAV가 보장하는 단조 증가성이 PCHIP에서도 유지된다 (PCHIP는 입력 monotonicity를 보존).

    Returns
    -------
    transform : callable. iso.transform과 같은 시그니처(raw -> calibrated).
    """
    x_knots = np.asarray(iso.X_thresholds_, dtype=float)
    y_knots = np.asarray(iso.y_thresholds_, dtype=float)
    # PAV 산출에서 X_thresholds_는 strictly increasing이지만, 방어적 dedupe.
    uniq_mask = np.concatenate([[True], np.diff(x_knots) > 0])
    x_knots = x_knots[uniq_mask]
    y_knots = y_knots[uniq_mask]
    if len(x_knots) < 2:
        # knot이 1개 이하면 PCHIP 불가능 → step 그대로 반환.
        def _fallback(x):
            return iso.transform(np.asarray(x, dtype=float))
        return _fallback, x_knots, y_knots

    pchip = PchipInterpolator(x_knots, y_knots, extrapolate=False)
    lo, hi = float(x_knots[0]), float(x_knots[-1])

    def _transform(x):
        x = np.asarray(x, dtype=float)
        x_clipped = np.clip(x, lo, hi)  # iso의 out_of_bounds='clip'과 같은 동작
        y_out = pchip(x_clipped)
        return np.clip(y_out, 0.0, None)  # y_min=0 강제

    return _transform, x_knots, y_knots


def fit_iso_tail_grid(train_unit, val_unit, test_unit, y_train_s, y_val_s, y_test_s):
    """unit-level 후처리 결과를 raw score로 보고 isotonic/tail 후보를 비교한다.

    hp/002 seed sweep 대비 차이:
    - `iso_kind ∈ {'step', 'pchip'}`이 grid 차원에 추가됨. PCHIP는 step의 knot 값을 그대로 두고 plateau만 smoothing.
    - `ISO_WEIGHTS`에 0.25, 0.5가 추가되어 raw 비중↑ 후보도 함께 탐색.

    train OOF raw -> train y로 fit하고, validation raw에는 transform만 적용한다.
    tail 강화는 raw score 상단부에만 추가 보정을 걸어 RMSE와 outlier 형성을 동시에 노린다.
    """
    raw_train = aligned_unit_pred(train_unit, y_train_s)
    raw_val = aligned_unit_pred(val_unit, y_val_s)
    raw_test = aligned_unit_pred(test_unit, y_test_s)
    y_train = y_train_s.values.astype(float)
    y_val = y_val_s.values.astype(float)
    y_test = y_test_s.values.astype(float)

    rows = []
    best = None

    def add_candidate(name, pred_train, pred_val, pred_test, params, iso_model=None, iso_kind=None, pchip_knots=None):
        nonlocal best
        pred_train = clip_nonneg(pred_train)
        pred_val = clip_nonneg(pred_val)
        pred_test = clip_nonneg(pred_test)
        val_stats = iqr_stats(pred_val, y_val)
        top_idx = int(np.argmax(pred_val))
        rec = {
            'name': name,
            'train_rmse': rmse(pred_train, y_train),
            'val_rmse': rmse(pred_val, y_val),
            # test_rmse는 모니터링용. 후보 선택(best 판정)에는 절대 쓰지 않는다 (val_rmse만 기준).
            'test_rmse': rmse(pred_test, y_test),
            'val_iqr_outliers': val_stats['n_upper_outliers'],
            'val_iqr_upper_fence': val_stats['upper_fence'],
            'val_max_pred': val_stats['max_pred'],
            'val_outlier_true_mean': val_stats['outlier_true_mean'],
            'val_outlier_true_max': val_stats['outlier_true_max'],
            'val_outlier_true_ge_q95': val_stats['outlier_true_ge_q95'],
            'val_top_pred_y_true': float(y_val[top_idx]),
            **params,
        }
        rows.append(rec)
        if best is None or rec['val_rmse'] < best['record']['val_rmse']:
            best = {
                'record': rec,
                'train_pred': pred_train,
                'val_pred': pred_val,
                'test_pred': pred_test,
                'iso_model': iso_model,
                'iso_kind': iso_kind,
                'pchip_knots': pchip_knots,
                'raw_train': raw_train,
                'raw_val': raw_val,
                'raw_test': raw_test,
            }

    # base 후보: iso/tail 없이 postprocess 결과 그대로.
    add_candidate(
        'base_postprocess', raw_train, raw_val, raw_test,
        {'uses_iso': False, 'iso_kind': 'none', 'iso_weight': 0.0,
         'tail_q': np.nan, 'tail_resid_q': np.nan,
         'tail_gain': 0.0, 'tail_power': np.nan, 'iqr_top_k': 0, 'tail_resid_scale': 0.0},
    )

    # PAV fit. step / pchip transform을 둘 다 미리 만들어 두고 itertools.product에서 룩업.
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0)
    iso.fit(raw_train, y_train)
    iso_train_step = iso.transform(raw_train)
    iso_val_step = iso.transform(raw_val)
    iso_test_step = iso.transform(raw_test)
    pchip_transform, pchip_x_knots, pchip_y_knots = build_iso_pchip_transform(iso)
    iso_train_pchip = pchip_transform(raw_train)
    iso_val_pchip = pchip_transform(raw_val)
    iso_test_pchip = pchip_transform(raw_test)

    iso_table = {
        'step':  (iso_train_step,  iso_val_step,  iso_test_step),
        'pchip': (iso_train_pchip, iso_val_pchip, iso_test_pchip),
    }

    for iso_kind, iso_weight, tail_q, tail_resid_q, tail_gain, tail_power, iqr_top_k in itertools.product(
        ISO_KINDS, ISO_WEIGHTS, TAIL_QS, TAIL_RESID_QS, TAIL_GAINS, TAIL_POWERS, IQR_TOP_KS
    ):
        iso_train_arr, iso_val_arr, iso_test_arr = iso_table[iso_kind]

        # iso_weight=1이면 순수 isotonic, 1보다 크면 isotonic 방향으로 더 강하게 당긴다.
        base_train = raw_train + iso_weight * (iso_train_arr - raw_train)
        base_val = raw_val + iso_weight * (iso_val_arr - raw_val)
        base_test = raw_test + iso_weight * (iso_test_arr - raw_test)

        # tail_start 이상 영역만 ramp. tail_resid_scale은 train tail의 양의 residual 분위수.
        tail_start = float(np.quantile(raw_train, tail_q))
        tail_hi = float(np.quantile(raw_train, 0.999))
        tail_denom = max(tail_hi - tail_start, 1e-12)
        tail_mask = raw_train >= tail_start
        if int(tail_mask.sum()) >= 3:
            resid = y_train[tail_mask] - base_train[tail_mask]
            tail_resid_scale = max(0.0, float(np.quantile(resid, tail_resid_q)))
        else:
            tail_resid_scale = 0.0

        def transform(raw, base, ts=tail_start, td=tail_denom, tp=tail_power, tg=tail_gain, trs=tail_resid_scale):
            ramp = np.clip((raw - ts) / td, 0.0, None) ** tp
            return base + tg * trs * ramp

        pred_train = transform(raw_train, base_train)
        pred_val = transform(raw_val, base_val)
        pred_test = transform(raw_test, base_test)

        # IQR outlier push는 y_true를 보지 않고 예측 rank/quantile만 사용 - validation leakage 방지.
        pred_train = push_top_k_to_iqr(pred_train, raw_train, iqr_top_k, IQR_MARGIN)
        pred_val = push_top_k_to_iqr(pred_val, raw_val, iqr_top_k, IQR_MARGIN)
        pred_test = push_top_k_to_iqr(pred_test, raw_test, iqr_top_k, IQR_MARGIN)

        name = f'iso{iso_kind}_w{iso_weight:g}_q{tail_q:g}_rq{tail_resid_q:g}_g{tail_gain:g}_p{tail_power:g}_iqr{iqr_top_k}'
        add_candidate(
            name, pred_train, pred_val, pred_test,
            {
                'uses_iso': True,
                'iso_kind': iso_kind,
                'iso_weight': float(iso_weight),
                'tail_q': float(tail_q),
                'tail_resid_q': float(tail_resid_q),
                'tail_gain': float(tail_gain),
                'tail_power': float(tail_power),
                'iqr_top_k': int(iqr_top_k),
                'tail_start': tail_start,
                'tail_denom': tail_denom,
                'tail_resid_scale': float(tail_resid_scale),
            },
            iso_model=iso,
            iso_kind=iso_kind,
            pchip_knots=(pchip_x_knots, pchip_y_knots) if iso_kind == 'pchip' else None,
        )

    cand = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    iqr12 = cand[cand['val_iqr_outliers'].between(1, 2)].copy()
    best_iqr12 = iqr12.iloc[0].to_dict() if len(iqr12) else None

    best['candidates'] = cand
    best['best_iqr12'] = best_iqr12
    best['raw_train'] = raw_train
    best['raw_val'] = raw_val
    best['raw_test'] = raw_test
    return best

## 4. HPO/ζ 공통 헬퍼 (folds·build_params·SPACE·ANCHOR_HP)

In [5]:
# ===== HPO/ζ 공통 헬퍼 (ζ 결정·HPO 둘 다 사용하므로 앞에 정의) =====
# unit-grouped folds: 같은 unit의 4 die가 train/valid에 섞이면 누수 -> unit ID 기준 분할.
def make_folds(seed, n_folds):
    units = y_train_unit_s.index.values
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=int(seed))
    return units, list(kf.split(units))


def build_params(hp, zeta):
    p = dict(hp)
    p['zeta'] = float(zeta)
    p['n_jobs'] = N_JOBS
    p['verbose'] = -1
    p['device'] = 'cpu'
    p['em_tol'] = 1e-7
    return p


def oof_unit_rmse(hp, zeta, n_folds, seed=42, want_arrays=False):
    """unit-grouped n-fold OOF die 예측 -> 순수 (1-π)μ -> mean 집계 -> unit RMSE.

    논문 예측 E[Y]=(1-π)μ 그대로(tau_pi 게이트 없음). HPO는 논문 모델 자체에 HP를 맞춘다.
    """
    units, folds = make_folds(seed, n_folds)
    n = len(X_train)
    oof_pi = np.full(n, np.nan)
    oof_mu = np.full(n, np.nan)
    for tr_u, vl_u in folds:
        trm = np.isin(uid_train_die, units[tr_u])
        vlm = np.isin(uid_train_die, units[vl_u])
        m = ZITboostGu(**build_params(hp, zeta), random_state=int(seed))
        m.fit(X_train[trm], y_train_die[trm])
        pi, mu, _ = m.predict_components(X_train[vlm])
        oof_pi[vlm] = pi
        oof_mu[vlm] = mu
    pred = np.clip((1.0 - oof_pi) * oof_mu, 0.0, None)     # 논문 예측 (1-π)μ (게이트 없음)
    unit = postprocess.aggregate(xs_train, pred, 'mean')   # plain ZIT die->unit = mean
    r = unit_rmse(unit, y_train_unit_s)
    if want_arrays:
        return r, oof_pi, oof_mu
    return r


# anchor HP: param1_005t17의 검증된 HP(n_em_iters만 속도 위해 15로 캡).
#   (1) ζ를 먼저 정할 때 이 HP로 적합, (2) HPO enqueue 첫 trial로도 사용.
ANCHOR_HP = {
    'n_em_iters': 15,
    'mu_n_estimators': 81, 'mu_learning_rate': 0.0042002, 'mu_num_leaves': 155,
    'mu_max_depth': 4, 'mu_min_child_samples': 123, 'mu_subsample': 0.546166,
    'mu_colsample_bytree': 0.541475, 'mu_reg_alpha': 2.1257e-05, 'mu_reg_lambda': 0.0181709,
    'pi_n_estimators': 387, 'pi_learning_rate': 0.0156186, 'pi_num_leaves': 111,
    'pi_max_depth': 22, 'pi_min_child_samples': 34,
    'phi_n_estimators': 127, 'phi_learning_rate': 0.0052176, 'phi_num_leaves': 46,
    'phi_max_depth': 4, 'phi_min_child_samples': 234,
}

# 경량 search space (param1_005t17 study 범위 참고, n_em_iters만 속도 위해 축소).
SPACE = {
    'n_em_iters':           ('int',  6,   15),
    'mu_n_estimators':      ('int',  50,  180),
    'mu_learning_rate':     ('flog', 0.0035, 0.012),
    'mu_num_leaves':        ('int',  80,  256),
    'mu_max_depth':         ('int',  3,   7),
    'mu_min_child_samples': ('int',  60,  220),
    'mu_subsample':         ('f',    0.4, 0.95),
    'mu_colsample_bytree':  ('f',    0.22, 0.85),
    'mu_reg_alpha':         ('flog', 1e-6, 5e-4),
    'mu_reg_lambda':        ('flog', 3e-3, 0.06),
    'pi_n_estimators':      ('int',  250, 550),
    'pi_learning_rate':     ('flog', 0.01, 0.045),
    'pi_num_leaves':        ('int',  70,  180),
    'pi_max_depth':         ('int',  8,   24),
    'pi_min_child_samples': ('int',  30,  90),
    'phi_n_estimators':     ('int',  90,  250),
    'phi_learning_rate':    ('flog', 0.004, 0.014),
    'phi_num_leaves':       ('int',  24,  110),
    'phi_max_depth':        ('int',  2,   6),
    'phi_min_child_samples':('int',  80,  260),
}


def suggest_hp(trial):
    hp = {}
    for k, (t, lo, hi) in SPACE.items():
        if t == 'int':
            hp[k] = trial.suggest_int(k, lo, hi)
        elif t == 'f':
            hp[k] = trial.suggest_float(k, lo, hi)
        elif t == 'flog':
            hp[k] = trial.suggest_float(k, lo, hi, log=True)
    return hp


## 5. [먼저] ζ 결정 — 논문 Algorithm 2 profile likelihood (anchor HP)

In [6]:
# ===== [먼저] ζ 결정 — 논문 Algorithm 2 profile likelihood =====
# HP는 아직 모르니 ANCHOR_HP로 ζ 그리드를 적합 -> val loglik 최대로 ζ* 선택.
# ζ는 HP에 둔감(데이터 양수부의 분포 모양)하므로 anchor HP로 골라도 안정적.
# 이렇게 정한 ζ*를 이후 HPO에서 고정으로 쓴다 (논문: ζ=바깥, GBT HP=안쪽).
y_val_die = xs_val[KEY_COL].map(y_val_unit_s).values.astype(np.float64)

zeta_rows = []
t_z = time.time()
for z in ZETA_GRID:
    m = ZITboostGu(**build_params(ANCHOR_HP, z), random_state=42)
    m.fit(X_train, y_train_die)                       # anchor HP로 전체 train 적합
    ll = float(m.score_loglik(X_val, y_val_die))      # Algorithm 2 목적함수 (EQL 로그우도)
    pi, mu, _ = m.predict_components(X_val)
    pred = np.clip((1.0 - pi) * mu, 0.0, None)        # 논문 예측 (1-π)μ (게이트 없음)
    vunit = postprocess.aggregate(xs_val, pred, 'mean')
    vr = unit_rmse(vunit, y_val_unit_s)
    zeta_rows.append({'zeta': float(z), 'loglik': ll, 'val_rmse': float(vr)})
    print(f'  zeta={z:.2f}: loglik={ll:.1f}, val_rmse={vr:.9f}')

zeta_df = pd.DataFrame(zeta_rows)
BEST_ZETA = float(zeta_df.loc[zeta_df['loglik'].idxmax(), 'zeta'])     # 논문: profile likelihood 최대
_zeta_by_rmse = float(zeta_df.loc[zeta_df['val_rmse'].idxmin(), 'zeta'])
print(f'[zeta*] profile-likelihood 선택 = {BEST_ZETA}  '
      f'(참고: val RMSE 최소 zeta = {_zeta_by_rmse}), elapsed={(time.time()-t_z)/60:.1f}min')
print(f'[다음] 이제 HPO를 ζ={BEST_ZETA} 고정으로 진행한다.')


  zeta=1.40: loglik=24993.2, val_rmse=0.005746418
  zeta=1.45: loglik=25610.6, val_rmse=0.005740820
  zeta=1.50: loglik=25929.7, val_rmse=0.005733155
  zeta=1.55: loglik=25815.7, val_rmse=0.005722775
  zeta=1.60: loglik=25053.8, val_rmse=0.005710292
[zeta*] profile-likelihood 선택 = 1.5  (참고: val RMSE 최소 zeta = 1.6), elapsed=63.2min
[다음] 이제 HPO를 ζ=1.5 고정으로 진행한다.


## 6. Optuna HPO (ζ* 고정, 외부 GBT HP만 탐색)

In [7]:
# ===== Optuna HPO (ζ=BEST_ZETA 고정, objective = OOF unit RMSE) =====
# make_folds/build_params/SPACE/suggest_hp/oof_unit_rmse/ANCHOR_HP는 앞 helpers 셀에 정의됨.
# ζ는 앞 셀에서 profile likelihood로 이미 결정(BEST_ZETA). 여기선 외부 GBT HP만 흔든다.
def objective(trial):
    hp = suggest_hp(trial)
    return oof_unit_rmse(hp, BEST_ZETA, HPO_N_FOLDS)


timeout_sec = max(300.0, (HPO_DEADLINE - datetime.now()).total_seconds())
print(f'[HPO] timeout={timeout_sec/3600:.2f}h, max_trials={HPO_MAX_TRIALS}, '
      f'folds={HPO_N_FOLDS}, zeta(fixed)={BEST_ZETA}')

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True, group=True),
)
# anchor: 검증된 HP(ANCHOR_HP)를 첫 trial로 enqueue -> TPE 워밍업 + 최소 1개 양질 trial 보장.
study.enqueue_trial(ANCHOR_HP)

t_hpo = time.time()


def _log_cb(study, trial):
    # trial마다 진행상황/현재 best 출력 (장시간 HPO 가시성 확보).
    try:
        best = study.best_value
    except ValueError:
        best = float('nan')
    v = trial.value if trial.value is not None else float('nan')
    print(f'  trial {trial.number}: rmse={v:.9f}  best={best:.9f}  '
          f'({(time.time()-t_hpo)/60:.1f}min, {len(study.trials)} done)')


study.optimize(objective, n_trials=HPO_MAX_TRIALS, timeout=timeout_sec,
               show_progress_bar=False, callbacks=[_log_cb])

BEST_HP = dict(study.best_params)
print(f'[HPO done] trials={len(study.trials)}, elapsed={(time.time()-t_hpo)/60:.1f}min, '
      f'best OOF unit RMSE={study.best_value:.9f}')
print(f'[best HP] {BEST_HP}')


[HPO] timeout=14.27h, max_trials=30, folds=3, zeta(fixed)=1.5
  trial 0: rmse=0.005517438  best=0.005517438  (21.4min, 1 done)
  trial 1: rmse=0.005508601  best=0.005508601  (29.0min, 2 done)
  trial 2: rmse=0.005521134  best=0.005508601  (42.2min, 3 done)
  trial 3: rmse=0.005504260  best=0.005504260  (53.1min, 4 done)
  trial 4: rmse=0.005514790  best=0.005504260  (66.8min, 5 done)
  trial 5: rmse=0.005507821  best=0.005504260  (92.5min, 6 done)
  trial 6: rmse=0.005509482  best=0.005504260  (110.1min, 7 done)
  trial 7: rmse=0.005516634  best=0.005504260  (136.2min, 8 done)
  trial 8: rmse=0.005516808  best=0.005504260  (193.8min, 9 done)
  trial 9: rmse=0.005515898  best=0.005504260  (266.0min, 10 done)
  trial 10: rmse=0.005502811  best=0.005502811  (329.9min, 11 done)
  trial 11: rmse=0.005504887  best=0.005502811  (360.5min, 12 done)
  trial 12: rmse=0.005503416  best=0.005502811  (384.2min, 13 done)
  trial 13: rmse=0.005501825  best=0.005501825  (413.7min, 14 done)
  trial 14:

## 7. 최종 5-fold OOF + 후처리 + isotonic/tail

In [8]:
# ===== 최종: best (HP, ζ)로 5-fold OOF -> 후처리 -> isotonic/tail =====
FINAL_PARAMS = build_params(BEST_HP, BEST_ZETA)
print(f'[final] zeta={BEST_ZETA}, n_folds={N_FOLDS}')
print(f'[final HP] {FINAL_PARAMS}')

units, folds = make_folds(42, N_FOLDS)
n_tr, n_vl, n_te = len(X_train), len(X_val), len(X_test)
oof_pi = np.full(n_tr, np.nan); oof_mu = np.full(n_tr, np.nan)
val_pi = np.zeros(n_vl); val_mu = np.zeros(n_vl)        # 성분 평균 (csv 진단용)
test_pi = np.zeros(n_te); test_mu = np.zeros(n_te)
val_pred = np.zeros(n_vl); test_pred = np.zeros(n_te)   # 예측 (1-π)μ 의 폴드 평균 (정석 앙상블)

t0 = time.time()
for i, (tr_u, vl_u) in enumerate(folds):
    trm = np.isin(uid_train_die, units[tr_u])
    vlm = np.isin(uid_train_die, units[vl_u])
    m = ZITboostGu(**FINAL_PARAMS, random_state=42 * 1009 + i)
    m.fit(X_train[trm], y_train_die[trm])
    pi, mu, _ = m.predict_components(X_train[vlm]); oof_pi[vlm] = pi; oof_mu[vlm] = mu
    pv, mv, _ = m.predict_components(X_val)
    val_pi += pv / N_FOLDS; val_mu += mv / N_FOLDS                  # 성분 평균(진단)
    val_pred += np.clip((1.0 - pv) * mv, 0.0, None) / N_FOLDS       # 예측 평균(앙상블)
    pt, mt, _ = m.predict_components(X_test)
    test_pi += pt / N_FOLDS; test_mu += mt / N_FOLDS
    test_pred += np.clip((1.0 - pt) * mt, 0.0, None) / N_FOLDS
    print(f'  fold {i+1}/{N_FOLDS} done, elapsed={time.time()-t0:.0f}s')

if np.isnan(oof_pi).any():
    raise RuntimeError('OOF에 NaN — fold 커버리지 확인 필요')

# die-level 최종 raw 예측 = 논문 예측 (1-π)μ (tau_pi 게이트 없음).
#   OOF(train)는 die마다 단일 fold라 성분에서 바로 계산.
#   val/test는 5개 fold 모델의 '예측'을 평균낸 val_pred/test_pred 사용 (성분 평균 아님 = 정석 앙상블).
oof_raw = np.clip((1.0 - oof_pi) * oof_mu, 0.0, None)
val_raw = np.clip(val_pred, 0.0, None)
test_raw = np.clip(test_pred, 0.0, None)

# 후처리(집계 mean 계열 + zero_clip) — 06 노트북 공통함수 재사용
pp_res = tune_unit_postprocess_train_val(
    xs_train, xs_val, xs_test,
    oof_raw, val_raw, test_raw,
    ys_input['train'], ys_input['validation'],
)
# isotonic/tail grid (step/pchip + iso_weight + tail) — 06 노트북 공통함수 재사용
cal = fit_iso_tail_grid(
    pp_res['final_train_unit'], pp_res['final_val_unit'], pp_res['final_test_unit'],
    y_train_unit_s, y_val_unit_s, y_test_unit_s,
)
rec = cal['record']
base_test_rmse = unit_rmse(pp_res['final_test_unit'], y_test_unit_s)
print(f'\n[최종 결과] base_val={pp_res["val_rmse_final"]:.9f} -> best_val={rec["val_rmse"]:.9f}, '
      f'test={rec["test_rmse"]:.9f} (base_test={base_test_rmse:.9f}), cal={rec["name"]}')


[final] zeta=1.5, n_folds=5
[final HP] {'n_em_iters': 11, 'mu_n_estimators': 98, 'mu_learning_rate': 0.0038262842716887723, 'mu_num_leaves': 110, 'mu_max_depth': 4, 'mu_min_child_samples': 67, 'mu_subsample': 0.6750108894729173, 'mu_colsample_bytree': 0.7329865174132766, 'mu_reg_alpha': 0.00024223986632685346, 'mu_reg_lambda': 0.0063740888888966585, 'pi_n_estimators': 491, 'pi_learning_rate': 0.042469388580162706, 'pi_num_leaves': 151, 'pi_max_depth': 20, 'pi_min_child_samples': 87, 'phi_n_estimators': 172, 'phi_learning_rate': 0.007976865396467386, 'phi_num_leaves': 37, 'phi_max_depth': 3, 'phi_min_child_samples': 123, 'zeta': 1.5, 'n_jobs': -1, 'verbose': -1, 'device': 'cpu', 'em_tol': 1e-07}
  fold 1/5 done, elapsed=611s
  fold 2/5 done, elapsed=1213s
  fold 3/5 done, elapsed=1823s
  fold 4/5 done, elapsed=2425s
  fold 5/5 done, elapsed=3027s
[Aggregation] RMSEs: {'mean': 0.005501, 'median': 0.0055, 'trimmed_mean': 0.0055, 'Q75': 0.005503, 'max': 0.005509}
[Aggregation] best=median 

## 8. 산출물 저장 + 요약

In [9]:
# ===== 산출물 저장 + 요약 =====
def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


summary = {
    'model': MODEL_NAME,
    'best_zeta': BEST_ZETA,
    'best_hp': BEST_HP,
    'hpo_best_oof_rmse': float(study.best_value),
    'hpo_n_trials': len(study.trials),
    'base_val_rmse': float(pp_res['val_rmse_final']),
    'val_rmse': float(rec['val_rmse']),
    'test_rmse': float(rec['test_rmse']),
    'base_test_rmse': float(base_test_rmse),
    'calibration_name': rec['name'],
    'postprocess_best_agg': pp_res['best_agg'],
    'postprocess_best_zero_clip': pp_res['best_zero_clip'],
    'zeta_grid_result': zeta_df.to_dict('records'),
    'n_features': len(feat_cols_clean),
    'pp_params': PP_PARAMS,
    'created_at': datetime.now().isoformat(timespec='seconds'),
}
with open(OUT_DIR / 'zit_gu_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=_json_default)

# die-level 예측 저장 (분석/앙상블용)
pd.DataFrame({
    KEY_COL: uid_val_die, DIE_KEY_COL: xs_val[DIE_KEY_COL].values,
    'position': xs_val['position'].values,
    'pi': val_pi, 'mu': val_mu, 'pred_raw': val_raw,
}).to_csv(OUT_DIR / 'val_die.csv', index=False)
pd.DataFrame({
    KEY_COL: uid_test_die, DIE_KEY_COL: xs_test[DIE_KEY_COL].values,
    'position': xs_test['position'].values,
    'pi': test_pi, 'mu': test_mu, 'pred_raw': test_raw,
}).to_csv(OUT_DIR / 'test_die.csv', index=False)
cal['candidates'].to_csv(OUT_DIR / 'calibration_candidates.csv', index=False)

print(f'[저장] {OUT_DIR}')
print('\n[ζ profile likelihood]')
display(zeta_df)
print('\n[요약]')
print(json.dumps(
    {k: summary[k] for k in
     ['best_zeta', 'hpo_best_oof_rmse',
      'base_val_rmse', 'val_rmse', 'test_rmse', 'calibration_name']},
    ensure_ascii=False, indent=2, default=_json_default))


[저장] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\zit_gu\zit_gu_optuna\run_0608_113938

[ζ profile likelihood]


,zeta,loglik,val_rmse
0,1.40,24993.241765,0.005746
1,1.45,25610.639403,0.005741
2,1.50,25929.664425,0.005733
3,1.55,25815.711198,0.005723
4,1.60,25053.777774,0.005710



[요약]
{
  "best_zeta": 1.5,
  "hpo_best_oof_rmse": 0.005501357419533067,
  "base_val_rmse": 0.005710029112471198,
  "val_rmse": 0.005698175824412313,
  "test_rmse": 0.008406620011944735,
  "calibration_name": "isostep_w1.25_q0.95_rq0.75_g0_p1_iqr0"
}
